# 5. Lasso Regression (L1 Regularization)
### Linear Regression with L1 Regularization (Sparse Solutions)

**Lasso** (Least Absolute Shrinkage and Selection Operator) adds an $L_1$ penalty proportional to the absolute values of coefficients, driving non-essential coefficients strictly to zero.

#### Key Concepts:
1. **Regularized Cost Function**:
   $$J(w) = \frac{1}{2m} \sum_{i=1}^m (y_i - \hat{y}_i)^2 + \alpha \sum_{j=1}^p |w_j|$$
2. **Sparsity & Automatic Feature Selection**:
   The geometry of the $L_1$ diamond constraint causes corners to intersect axes, zeroing out redundant weights.
3. **Coordinate Descent**:
   Because the $L_1$ penalty is non-differentiable at $w=0$, Lasso uses coordinate descent to optimize parameters.


In [ ]:
# Step 1: Import Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso, LassoCV
from sklearn.metrics import r2_score, mean_squared_error


In [ ]:
# Step 2: Load and Prepare Data
diabetes = load_diabetes()
X = diabetes.data
y = diabetes.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:
# Step 3: LassoCV for Automated Optimal Alpha Selection
lasso_cv = LassoCV(alphas=np.logspace(-3, 1, 50), cv=5, random_state=42)
lasso_cv.fit(X_train_scaled, y_train)

y_pred = lasso_cv.predict(X_test_scaled)
print(f"Optimal Alpha: {lasso_cv.alpha_:.4f}")
print(f"Lasso Test R² Score: {r2_score(y_test, y_pred):.4f}")
print(f"Lasso Test RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")

# Check sparsity (zeroed features)
coef_df = pd.DataFrame({"Feature": diabetes.feature_names, "Coefficient": lasso_cv.coef_})
print("
Lasso Coefficients (Zero = Dropped Feature):
", coef_df)
print(f"Number of retained features: {np.sum(lasso_cv.coef_ != 0)} / {len(diabetes.feature_names)}")


In [ ]:
# Step 4: Lasso Regularization Path
alphas = np.logspace(-3, 1, 50)
coefs = []
for a in alphas:
    m = Lasso(alpha=a, max_iter=2000, random_state=42)
    m.fit(X_train_scaled, y_train)
    coefs.append(m.coef_)

plt.figure(figsize=(10, 5))
plt.plot(alphas, coefs)
plt.xscale('log')
plt.xlabel("Alpha (Regularization Strength)")
plt.ylabel("Coefficients")
plt.title("Lasso Coefficients Path (Notice features dropping to exactly 0)")
plt.axvline(lasso_cv.alpha_, color='k', linestyle='--', label=f'Best Alpha: {lasso_cv.alpha_:.3f}')
plt.legend()
plt.grid(True)
plt.show()


### Summary
- **Strengths**: Performs automatic feature selection, outputs sparse, interpretable models.
- **Weaknesses**: If features are highly correlated, Lasso tends to arbitrarily select one and zero out the rest; limited when $p > n$.
